# 04 — Paired GLueCoS Evaluation

Run this notebook only after both selected MLM checkpoints are available. It fine-tunes the BPE control and language-conditioned probabilistic-fusion model on the same local GLueCoS splits for every configured seed. All training, inference, checkpointing, and prediction serialization live in `src.training.finetune`; this notebook is only a reproducible local entry point.

`runtime.mode` controls behavior: `train` creates missing per-seed results, `resume` reuses valid saved results and completes missing ones, and `evaluate` requires all saved outputs without fitting a model.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    configured_root = os.environ.get('PROJECT_ROOT')
    starting_points = [Path(configured_root)] if configured_root else [Path.cwd()]
    for starting_point in starting_points:
        resolved_start = starting_point.expanduser().resolve()
        for candidate in (resolved_start, *resolved_start.parents):
            if (candidate / 'configs' / 'config.yaml').is_file() and (candidate / 'INSTRUCTIONS.md').is_file():
                return candidate
    raise FileNotFoundError('Could not locate PROJECT_ROOT. Launch Jupyter from the repository root or set PROJECT_ROOT.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


## 1. Load the saved configuration

The notebook loads the same configuration and seed list recorded by pretraining. It does not change tokenizers, MLM checkpoints, task splits, or fine-tuning hyperparameters.

In [ ]:
import json
import random

import numpy as np
import torch
import yaml

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'config.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
required_sections = {'runtime', 'paths', 'model', 'training', 'finetuning', 'evaluation', 'experiment'}
missing_sections = required_sections.difference(config)
if missing_sections:
    raise KeyError(f'Missing configuration sections: {sorted(missing_sections)}')

seed = int(config['training']['seed'])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

print(json.dumps({
    'runtime_mode': config['runtime']['mode'],
    'experiment_id': config['experiment']['id'],
    'evaluation_seeds': config['evaluation']['seeds'],
    'gluecos_repositories': config['data_download']['gluecos'],
}, indent=2))


## 2. Verify the selected paired MLM checkpoints

Fine-tuning must use the validation-selected `best` checkpoint from each completed pretrained arm. This check is deliberately strict: it prevents an incomplete, final-only, or unmatched model from entering the GLueCoS comparison.

In [ ]:
CHECKPOINT_ROOT = PROJECT_ROOT / config['paths']['checkpoints']
BPE_SELECTED_CHECKPOINT = CHECKPOINT_ROOT / 'bpe_model' / 'best'
PROBABILISTIC_SELECTED_CHECKPOINT = CHECKPOINT_ROOT / 'probabilistic_model' / 'best'


def complete_checkpoint(path: Path) -> bool:
    return (path / 'trainer_state.pt').is_file() and (path / 'config.json').is_file()


checkpoint_status = {
    'bpe': {'path': str(BPE_SELECTED_CHECKPOINT), 'complete': complete_checkpoint(BPE_SELECTED_CHECKPOINT)},
    'probabilistic': {'path': str(PROBABILISTIC_SELECTED_CHECKPOINT), 'complete': complete_checkpoint(PROBABILISTIC_SELECTED_CHECKPOINT)},
}
print(json.dumps(checkpoint_status, indent=2))
if not all(item['complete'] for item in checkpoint_status.values()):
    print('Downstream evaluation is pending until both selected MLM checkpoints are complete.')


## 3. Run or resume paired GLueCoS fine-tuning and inference

For each repository and configured seed, the pipeline uses identical task heads, splits, optimizer, scheduler, epoch budget, and validation checkpoint rule for both arms. It saves selected task checkpoints, epoch logs, aligned test predictions, labels, and metrics under configured output directories. Existing complete per-seed result files are validated and reused.

In [ ]:
from src.training.finetune import run_gluecos_finetuning

if not all(item['complete'] for item in checkpoint_status.values()):
    raise FileNotFoundError('Both selected MLM checkpoints are required before GLueCoS fine-tuning.')

finetuning_results = run_gluecos_finetuning(
    PROJECT_ROOT,
    config,
    BPE_SELECTED_CHECKPOINT,
    PROBABILISTIC_SELECTED_CHECKPOINT,
)
print({repository: sorted(result['seeds']) for repository, result in finetuning_results.items()})


## 4. Inspect saved local inference outputs

The displayed examples come from the persisted paired prediction files rather than a new, ad-hoc inference pass. This keeps the later bootstrap and seed-level tests tied to exactly the predictions produced by the selected fine-tuning checkpoints.

In [ ]:
from src.data.gluecos_loader import safe_dataset_name

RESULT_ROOT = PROJECT_ROOT / config['paths']['results']
inspection_seed = str(int(config['evaluation']['seeds'][0]))
for repository, task_result in finetuning_results.items():
    task = task_result['task']
    result_path = RESULT_ROOT / task / safe_dataset_name(repository) / f'seed_{inspection_seed}.json'
    payload = json.loads(result_path.read_text(encoding='utf-8'))
    print(f'\n{repository} | task={task} | seed={inspection_seed}')
    for arm in ('bpe', 'probabilistic'):
        arm_result = payload[arm]
        preview = list(zip(arm_result['labels'][:10], arm_result['predictions'][:10]))
        print(f"  {arm}: metrics={arm_result['metrics']}\n    label/prediction preview={preview}")

experiment_summary = PROJECT_ROOT / config['paths']['experiments'] / config['experiment']['id'] / 'downstream_metrics.json'
print(f'\nSaved paired downstream summary: {experiment_summary}')


## Next step

After all five seeds have saved for every task, run `07_experiment_summary_and_statistics.ipynb`. It verifies aligned references, computes seed summaries and paired tests, performs the paired bootstrap comparison, and writes the final JSON and Markdown reports.